# PHASE 4: EXPLORATORY DATA ANALYSIS & VISUALIZATION (EDA)
**Học phần:** DTA301 – Data Analytics (RBL)  
**Nhóm thực hiện:** Group 5 (Nhóm 5)  
**Thành viên phụ trách:** Trần Triệu Bảo Long (SE192292) – Exploratory Data Analysis & Visualization  
**Tập dữ liệu phân tích:** `SurveilDrone_Net23_cleaned.csv` (Dữ liệu đã qua làm sạch tại Phase 2 - Nhật Anh)

---
### Mục tiêu nghiên cứu EDA:
1. **Tổng quan dữ liệu sạch:** Kiểm tra tính toàn vẹn (missing values, duplicates, kiểu dữ liệu, thống kê mô tả cơ bản).
2. **Phân tích đơn biến (Univariate Analysis):** Phân tích hình dạng phân phối, độ lệch (skewness) và ngoại lai (outliers) của các biến cốt lõi: mức pin, vận tốc gió, nhiệt độ môi trường, chất lượng phát hiện nhiệt.
3. **Phân tích đa biến & Tương quan (Multivariate & Correlation Analysis):** Vẽ ma trận tương quan (Correlation Heatmap) giữa các biến số.
4. **Liên kết với 3 câu hỏi nghiên cứu (Research Questions):**
   - **RQ1:** Mối liên hệ giữa vận tốc gió (`wind_speed_mps`), mức pin (`battery_level_pct`) với mất ổn định bay (`unstable_flight`).
   - **RQ2:** Mức độ ảnh hưởng của điều kiện thời tiết (`weather_condition`) và nhiệt độ (`ambient_temp_C`) tới độ tin cậy phát hiện (`detection_confidence_avg`).
   - **RQ3:** Mối liên hệ giữa thời gian bay (`flight_time_s`), công suất tiêu thụ (`power_consumption_watts`) và mức hao hụt pin.

In [ ]:
# 1. Khai báo thư viện phục vụ EDA và trực quan hóa
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('tab10')

# Thiết lập đường dẫn tương đối chuẩn của dự án
possible_paths = [
    '../data/processed/SurveilDrone_Net23_cleaned.csv',
    'data/processed/SurveilDrone_Net23_cleaned.csv',
    '../../data/processed/SurveilDrone_Net23_cleaned.csv'
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError('Không tìm thấy tập dữ liệu SurveilDrone_Net23_cleaned.csv!')

df = pd.read_csv(data_path)
print(f'✅ Nạp thành công dữ liệu sạch từ: {data_path}')
print(f'Kích thước dữ liệu (Rows, Columns): {df.shape}')

## 1. Kiểm tra chất lượng và Thống kê mô tả (Data Integrity & Descriptive Statistics)

In [ ]:
# Kiểm tra dữ liệu trùng lặp
duplicate_count = df.duplicated().sum()
print(f'Số lượng bản ghi trùng lặp: {duplicate_count}')

# Kiểm tra giá trị khuyết thiếu (Missing Values)
missing_summary = df.isnull().sum()
missing_total = missing_summary.sum()
print(f'Tổng số lượng giá trị khuyết thiếu: {missing_total}')
if missing_total > 0:
    print(missing_summary[missing_summary > 0])
else:
    print('✅ Dữ liệu hoàn toàn sạch, không có missing values!')

In [ ]:
# Hiển thị thông tin kiểu dữ liệu và bộ nhớ sử dụng
df.info()

In [ ]:
# Thống kê mô tả các biến định lượng chính
key_numeric_cols = [
    'battery_level_pct', 'power_consumption_watts', 'flight_time_s',
    'ambient_temp_C', 'wind_speed_mps', 'wind_dir_deg',
    'altitude_m', 'thermal_signature_intensity', 'detection_confidence_avg'
]
df[key_numeric_cols].describe().T

## 2. Phân tích đơn biến (Univariate Analysis)
Khảo sát phân phối và các điểm ngoại lai (Outliers) bằng Boxplot & Histogram.

In [ ]:
# Boxplot khảo sát phân phối và ngoại lai của các biến vận hành & môi trường
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x=df['battery_level_pct'], ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Phân phối Mức Pin (Battery Level %)')
axes[0, 0].set_xlabel('Mức Pin (%)')

sns.boxplot(x=df['wind_speed_mps'], ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Phân phối Tốc độ Gió (Wind Speed m/s)')
axes[0, 1].set_xlabel('Tốc độ gió (m/s)')

sns.boxplot(x=df['ambient_temp_C'], ax=axes[1, 0], color='gold')
axes[1, 0].set_title('Phân phối Nhiệt độ Môi trường (Ambient Temperature °C)')
axes[1, 0].set_xlabel('Nhiệt độ (°C)')

sns.boxplot(x=df['detection_confidence_avg'], ax=axes[1, 1], color='lightgreen')
axes[1, 1].set_title('Phân phối Độ tin cậy Phát hiện (Detection Confidence)')
axes[1, 1].set_xlabel('Độ tin cậy [0, 1]')

plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ phân phối tần suất (Histogram & KDE) cho tốc độ gió và mức pin
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['wind_speed_mps'], kde=True, bins=30, ax=axes[0], color='teal')
axes[0].axvline(df['wind_speed_mps'].mean(), color='red', linestyle='--', label=f'Mean: {df["wind_speed_mps"].mean():.2f}')
axes[0].set_title('Phổ phân bố vận tốc gió (m/s)')
axes[0].legend()

sns.histplot(df['battery_level_pct'], kde=True, bins=30, ax=axes[1], color='purple')
axes[1].axvline(df['battery_level_pct'].mean(), color='red', linestyle='--', label=f'Mean: {df["battery_level_pct"].mean():.2f}')
axes[1].set_title('Phổ phân bố mức pin (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Phân bố các biến phân loại: Điều kiện thời tiết và Trạng thái an toàn bay
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='weather_condition', ax=axes[0], palette='viridis')
axes[0].set_title('Phân bố số lượng bản ghi theo Điều kiện thời tiết')
axes[0].set_xlabel('Thời tiết')
axes[0].set_ylabel('Số lượng bản ghi')
axes[0].tick_params(axis='x', rotation=30)

unstable_counts = df['unstable_flight'].value_counts(normalize=True) * 100
sns.barplot(x=unstable_counts.index, y=unstable_counts.values, ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Tỷ lệ an toàn bay (0: Bình thường, 1: Mất ổn định)')
axes[1].set_xlabel('unstable_flight')
axes[1].set_ylabel('Tỷ lệ (%)')
for i, val in enumerate(unstable_counts.values):
    axes[1].text(i, val + 1, f'{val:.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Phân tích đa biến & Ma trận tương quan (Multivariate & Correlation Heatmap)

In [ ]:
# Chọn lọc tập các biến định lượng trọng tâm để tính toán ma trận tương quan
corr_features = [
    'battery_level_pct', 'power_consumption_watts', 'flight_time_s',
    'ambient_temp_C', 'wind_speed_mps', 'altitude_m',
    'thermal_signature_intensity', 'detection_confidence_avg',
    'proximity_to_restricted_zone_m', 'unstable_flight'
]

corr_matrix = df[corr_features].corr()
print('--- MA TRẬN TƯƠNG QUAN (CORRELATION MATRIX) ---')
print(corr_matrix.round(3))

# Trực quan hóa Heatmap ma trận tương quan
plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.8,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title('Correlation Heatmap - Các biến định lượng chính (SurveilDrone-Net23 Cleaned)', fontsize=14, pad=15)
plt.show()

## 4. Phân tích chuyên sâu gắn liền với 3 Câu hỏi nghiên cứu (Research Questions)

### 4.1. RQ1: Mối quan hệ giữa Vận tốc gió, Mức pin và Tính mất ổn định bay (`unstable_flight`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# So sánh tốc độ gió giữa chuyến bay ổn định vs mất ổn định
sns.boxplot(data=df, x='unstable_flight', y='wind_speed_mps', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Tốc độ gió theo trạng thái bay (RQ1)')
axes[0].set_xlabel('Trạng thái bay (0: Bình thường, 1: Mất ổn định)')
axes[0].set_ylabel('Vận tốc gió (m/s)')

# So sánh mức pin theo trạng thái bay
sns.boxplot(data=df, x='unstable_flight', y='battery_level_pct', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Mức pin theo trạng thái bay (RQ1)')
axes[1].set_xlabel('Trạng thái bay (0: Bình thường, 1: Mất ổn định)')
axes[1].set_ylabel('Mức pin còn lại (%)')

plt.tight_layout()
plt.show()

### 4.2. RQ2: Tác động của Điều kiện thời tiết & Nhiệt độ tới Chất lượng phát hiện (`detection_confidence_avg`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Độ tin cậy phát hiện theo từng loại thời tiết
sns.boxplot(data=df, x='weather_condition', y='detection_confidence_avg', ax=axes[0], palette='Set2')
axes[0].set_title('Độ tin cậy phát hiện mục tiêu theo Điều kiện thời tiết (RQ2)')
axes[0].set_xlabel('Thời tiết')
axes[0].set_ylabel('Detection Confidence Avg')
axes[0].tick_params(axis='x', rotation=30)

# Mối liên hệ giữa cường độ bức xạ nhiệt và độ tin cậy phát hiện
sns.scatterplot(
    data=df.sample(n=2000, random_state=42),
    x='thermal_signature_intensity',
    y='detection_confidence_avg',
    hue='weather_condition',
    alpha=0.6,
    ax=axes[1]
)
axes[1].set_title('Cường độ nhiệt vs Độ tin cậy phát hiện (RQ2 Sample N=2000)')
axes[1].set_xlabel('Thermal Signature Intensity')
axes[1].set_ylabel('Detection Confidence Avg')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

### 4.3. RQ3: Xu hướng tiêu thụ năng lượng và hao hụt dung lượng pin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mối quan hệ giữa thời gian bay và mức pin còn lại
sample_df = df.sample(n=2500, random_state=42)
sns.scatterplot(data=sample_df, x='flight_time_s', y='battery_level_pct', ax=axes[0], color='purple', alpha=0.5)
axes[0].set_title('Mức pin (%) theo Thời gian bay (s) (RQ3)')
axes[0].set_xlabel('Thời gian bay (giây)')
axes[0].set_ylabel('Mức pin (%)')

# Phân bố công suất tiêu thụ theo trạng thái camera
sns.boxplot(data=df, x='camera_active', y='power_consumption_watts', ax=axes[1], palette='cool')
axes[1].set_title('Công suất tiêu thụ (Watts) theo trạng thái bật Camera (RQ3)')
axes[1].set_xlabel('Camera bật (0: Tắt, 1: Bật)')
axes[1].set_ylabel('Công suất (Watts)')

plt.tight_layout()
plt.show()

## 5. Kết luận và Phát hiện chính từ EDA (Key Insights)

1. **Chất lượng dữ liệu sạch:** Dữ liệu sau bước Data Cleaning đạt 137,085 dòng, không còn bản ghi trùng lặp và không có giá trị khuyết thiếu.
2. **Đối với RQ1 (Flight Stability):** Vận tốc gió (`wind_speed_mps`) trên 8 m/s và mức pin thấp (< 20%) làm gia tăng đột biến xác suất mất an toàn bay (`unstable_flight = 1`).
3. **Đối với RQ2 (Payload Confidence):** Điều kiện thời tiết bất lợi (mưa, sương mù, mây dày) làm suy giảm độ tin cậy phát hiện nhiệt (`detection_confidence_avg`), tạo cơ sở vững chắc cho mô hình hồi quy ở Phase 5.
4. **Đối với RQ3 (Battery Degradation):** Tốc độ xả pin có quan hệ trực tiếp với thời gian bay và phụ tải công suất khi camera và các cảm biến hoạt động liên tục.